
# TrustCV Deep Learning Showcase (No Manual CV)
This notebook demonstrates how to evaluate **deep learning** models with:

- **`TrustCVValidator`** (leakage/balance + bootstrapped CIs)
- **`UniversalCVRunner`** (fold-by-fold training + per-fold indices/predictions)
- **`KerasSkWrap`** from **`trustcv.frameworks.tensorflow_sklearn`** (Keras ↔ sklearn interface)

## Clinical story
We simulate patient-level data with repeated visits:

- **Tabular clinical variables**: age, BMI, systolic BP  
- **Imaging biomarkers**: LV ejection fraction (EF), LV mass index  
- **ECG/echo flags** *(multi-label, C=3)*: AF, LVH, conduction abnormality  
- Outcomes:
  - **Main task**: 1-year MACE risk *(binary classification)*  
  - **Secondary**: future EF drop *(regression)*  

We keep the dataset synthetic so it runs everywhere (Colab/local) and avoids any real patient data.


In [ ]:

# --- Install (Colab only) ---
# !pip -q install trustcv==1.0.5 tensorflow scikit-learn pandas matplotlib

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, GroupKFold, KFold
from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    precision_score, recall_score, f1_score,
    mean_squared_error, r2_score
)

import trustcv
from trustcv import TrustCVValidator, UniversalCVRunner
from trustcv.splitters.grouped import StratifiedGroupKFold
from trustcv.frameworks.tensorflow_sklearn import KerasSkWrap, KerasRegressorWrap

SEED = 42
tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)

print("trustcv version:", trustcv.__version__)
print("tf version:", tf.__version__)
print("KerasSkWrap:", KerasSkWrap)


In [ ]:

# -------------------------
# Step 1) Simulate "medical" dataset (with patient groups + repeated visits)
# -------------------------
N_patients = 200
visits_per_patient = 4
N = N_patients * visits_per_patient

patient_ids = np.repeat(np.arange(N_patients), visits_per_patient)

# Clinical features
age = np.random.normal(65, 10, N)
bmi = np.random.normal(27, 4, N)
sbp = np.random.normal(135, 15, N)

# Imaging biomarkers
ef = np.random.normal(55, 8, N)
lvmass = np.random.normal(90, 20, N)

# Multi-label ECG/echo flags (C=3)
af_flag   = (np.random.rand(N) < 0.25).astype(int)
lvh_flag  = (np.random.rand(N) < 0.30).astype(int)
cond_flag = (np.random.rand(N) < 0.20).astype(int)
y_multi = np.stack([af_flag, lvh_flag, cond_flag], axis=1).astype("int32")

# Binary outcome: 1y MACE risk (depends on age, EF, flags + patient-level effect)
patient_risk = np.random.normal(0, 0.7, N_patients)  # latent patient effect
logit = (
    0.03*(age-60)
    -0.05*(ef-55)
    +0.8*af_flag +0.5*lvh_flag +0.6*cond_flag
    + patient_risk[patient_ids]
)
prob = 1/(1+np.exp(-logit))
y_main = (np.random.rand(N) < prob).astype("int32")

# Regression outcome: future EF drop
y_future_ef_drop = (
    np.maximum(0, 10 - 0.10*(ef-55)) + 2*af_flag + np.random.normal(0, 2, N)
).astype("float32")

# Assemble multi-input X representations
X_tab = np.stack([age, bmi, sbp], axis=1).astype("float32")
X_img = np.stack([ef, lvmass], axis=1).astype("float32")

X_single = np.concatenate([X_tab, X_img], axis=1)  # simple single-array input
X_dict = {"clinical": X_tab, "imaging": X_img}     # dict input
X_list = [X_tab, X_img]                            # list input

df_preview = pd.DataFrame({
    "patient_id": patient_ids,
    "age": age, "bmi": bmi, "sbp": sbp,
    "ef": ef, "lvmass": lvmass,
    "af": af_flag, "lvh": lvh_flag, "cond": cond_flag,
    "mace_1y": y_main,
    "future_ef_drop": y_future_ef_drop
})
display(df_preview.head(8))

print("Shapes:")
print("  X_single:", X_single.shape)
print("  X_tab   :", X_tab.shape, "X_img:", X_img.shape)
print("  y_main  :", y_main.shape, "y_multi:", y_multi.shape, "y_future:", y_future_ef_drop.shape)
print("  #patients:", len(np.unique(patient_ids)), "N:", len(patient_ids))



## Step 2) Make complex inputs **indexable** (no manual CV loops)

`UniversalCVRunner` and the sklearn-style splitters expect `X[train_idx]` / `y[val_idx]` to work.
NumPy arrays already support this.  
For **dict/list inputs** and **dict outputs**, we wrap them in small containers that implement:

- `__len__()` → number of samples
- `__getitem__(idx)` → returns sliced structure with the same type

This keeps the notebook **manual-loop free** while still demonstrating multi-input/multi-output Keras models.


In [ ]:

class IndexableDict:
    """Wrap a dict of arrays so X[idx] returns dict of arrays sliced by idx."""
    def __init__(self, data: dict):
        self.data = data
        # infer sample dimension from first key
        first_key = next(iter(data.keys()))
        self.n = int(data[first_key].shape[0])

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.data.items()}


class IndexableList:
    """Wrap a list/tuple of arrays so X[idx] returns list of arrays sliced by idx."""
    def __init__(self, data):
        self.data = list(data)
        self.n = int(self.data[0].shape[0])

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        return [v[idx] for v in self.data]


class IndexableTFDataset:
    """Wrap (X, y[, sample_weight]) arrays; slicing returns a tf.data.Dataset subset."""
    def __init__(self, X, y, sample_weight=None, batch_size=64, shuffle=False):
        self.X = X
        self.y = y
        self.sw = sample_weight
        self.batch_size = batch_size
        self.shuffle = shuffle
        # infer n from X (supports IndexableDict/IndexableList or np.ndarray)
        self.n = len(X)

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        Xb = self.X[idx]
        yb = self.y[idx]
        if self.sw is None:
            ds = tf.data.Dataset.from_tensor_slices((Xb, yb))
        else:
            swb = self.sw[idx]
            ds = tf.data.Dataset.from_tensor_slices((Xb, yb, swb))
        if self.shuffle:
            ds = ds.shuffle(min(2048, len(yb)), seed=SEED, reshuffle_each_iteration=True)
        ds = ds.batch(self.batch_size).prefetch(tf.data.AUTOTUNE)
        return ds

# Wrap complex structures
X_dict_idx = IndexableDict(X_dict)
X_list_idx = IndexableList(X_list)

y_multi_idx = y_multi  # already np.ndarray
y_main_idx = y_main    # already np.ndarray
y_future_idx = y_future_ef_drop  # already np.ndarray

# Multi-output y as dict (for multi-output model training)
y_multiout = IndexableDict({"mace": y_main.astype("int32"), "ef_drop": y_future_ef_drop.astype("float32")})

print("Indexable checks:")
print("  len(X_dict_idx) =", len(X_dict_idx), " | clinical slice shape:", X_dict_idx[np.arange(5)]["clinical"].shape)
print("  len(X_list_idx) =", len(X_list_idx), " | list slice[0] shape:", X_list_idx[np.arange(5)][0].shape)
print("  len(y_multiout) =", len(y_multiout), " | keys:", list(y_multiout.data.keys()))



## Step 3) TrustCVValidator: show why **group-aware CV** matters (leakage detection)

We intentionally run a **bad** configuration (groups ignored) and a **good** configuration (grouped splitter).

Expected:
- Bad config → `Leakage Check: FAILED` (patients appear in both train and test)
- Good config → `Leakage Check: PASSED`


In [ ]:

# -------------------------
# 3A) Keras binary model factory (single-array input)
# -------------------------
def build_mace_mlp(input_shape=(5,)):
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="binary_crossentropy",
                  metrics=["accuracy"])
    return model

# -------------------------
# 3B) BAD: StratifiedKFold ignores groups -> leakage likely with repeated patient visits
# -------------------------
bad = TrustCVValidator(
    method="stratified_kfold",
    n_splits=5,
    shuffle=True,
    random_state=SEED,
    check_leakage=True,
    check_balance=True,
)

res_bad = bad.validate(
    model=KerasSkWrap(build_fn=lambda input_shape=None, n_classes=None: build_mace_mlp(input_shape=(X_single.shape[1],)),
                      epochs=3, batch_size=64, verbose=0, task="binary"),
    X=X_single,
    y=y_main,
    groups=patient_ids,   # will be ignored by StratifiedKFold inside sklearn
)

print(res_bad.summary())

# -------------------------
# 3C) GOOD: StratifiedGroupedKFold (group-aware stratified splitting)
# -------------------------
good = TrustCVValidator(
    method="stratified_grouped_kfold",
    n_splits=5,
    shuffle=True,
    random_state=SEED,
    check_leakage=True,
    check_balance=True,
)

res_good = good.validate(
    model=KerasSkWrap(build_fn=lambda input_shape=None, n_classes=None: build_mace_mlp(input_shape=(X_single.shape[1],)),
                      epochs=3, batch_size=64, verbose=0, task="binary"),
    X=X_single,
    y=y_main,
    groups=patient_ids,
)

print(res_good.summary())



## Step 4) UniversalCVRunner: fold-by-fold metrics (no bootstrap aggregation)

`UniversalCVRunner` returns a `CVResults` object containing:
- per-fold `scores`
- per-fold `indices` (train/test index arrays)
- per-fold `predictions` / `probabilities` (when available)

We compute fold metrics ourselves to support:
- multilabel outputs
- regression
- multi-output models


In [ ]:

# -------------------------
# 4A) Runner on binary MACE model (single-array input)
# -------------------------
cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)
runner = UniversalCVRunner(cv_splitter=cv, framework="sklearn", verbose=1)

mace_clf = KerasSkWrap(
    build_fn=lambda input_shape=None, n_classes=None: build_mace_mlp(input_shape=(X_single.shape[1],)),
    epochs=3,
    batch_size=64,
    verbose=0,
    task="binary",
    threshold=0.5,
)

results = runner.run(model=mace_clf, data=(X_single, y_main), groups=patient_ids)

# fold-by-fold metrics computed from stored indices/predictions/probabilities
fold_rows = []
for i, (tr_idx, te_idx) in enumerate(results.indices):
    y_true = y_main[te_idx]
    y_prob = results.probabilities[i]
    # KerasSkWrap binary proba is (n,2): [:,1] is positive class
    p1 = y_prob[:, 1] if y_prob.ndim == 2 else y_prob
    y_pred = (p1 >= 0.5).astype(int)
    fold_rows.append({
        "fold": i+1,
        "accuracy": accuracy_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, p1),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "n_test": len(te_idx),
        "n_patients_test": len(np.unique(patient_ids[te_idx])),
    })

df_folds = pd.DataFrame(fold_rows)
display(df_folds)

print("Mean +/- std")
print(df_folds[["accuracy","roc_auc","precision","recall","f1"]].agg(["mean","std"]))



## Step 5) Multi-label classification (C=3) with KerasSkWrap

Task: predict the **three ECG/echo flags** (AF/LVH/Conduction).  
Model output: `sigmoid` with `C=3` columns.  
We compute **micro-F1** and **macro-F1** per fold.

Note: sklearn's `StratifiedKFold` does **not** support multilabel stratification; here we use `GroupKFold` for patient-safe splits.


In [ ]:

def build_multilabel_mlp(input_dim=5, C=3):
    inp = layers.Input(shape=(input_dim,))
    x = layers.Dense(32, activation="relu")(inp)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(16, activation="relu")(x)
    out = layers.Dense(C, activation="sigmoid")(x)
    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="binary_crossentropy")
    return model

cv_g = GroupKFold(n_splits=3)
runner_ml = UniversalCVRunner(cv_splitter=cv_g, framework="sklearn", verbose=1)

ml_clf = KerasSkWrap(
    build_fn=lambda input_shape=None, n_classes=None: build_multilabel_mlp(input_dim=X_single.shape[1], C=3),
    epochs=3, batch_size=64, verbose=0,
    task="multilabel",
    threshold=0.5,
)

ml_results = runner_ml.run(model=ml_clf, data=(X_single, y_multi_idx), groups=patient_ids)

rows = []
for i, (tr, te) in enumerate(ml_results.indices):
    y_true = y_multi_idx[te]
    y_pred = ml_results.predictions[i]  # expected (n,3) binary
    rows.append({
        "fold": i+1,
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "label_prev_AF": y_true[:,0].mean(),
        "label_prev_LVH": y_true[:,1].mean(),
        "label_prev_COND": y_true[:,2].mean(),
    })
df_ml = pd.DataFrame(rows)
display(df_ml)
print(df_ml[["micro_f1","macro_f1"]].agg(["mean","std"]))



## Step 6) Regression (future EF drop) with KerasRegressorWrap

We predict a continuous outcome and compute:
- RMSE
- R²


In [ ]:

def build_reg_mlp(input_dim=5):
    inp = layers.Input(shape=(input_dim,))
    x = layers.Dense(32, activation="relu")(inp)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(16, activation="relu")(x)
    out = layers.Dense(1, activation="linear")(x)
    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
    return model

cv_g = GroupKFold(n_splits=3)
runner_reg = UniversalCVRunner(cv_splitter=cv_g, framework="sklearn", verbose=1)

reg_model = KerasRegressorWrap(
    build_fn=lambda input_shape=None, n_classes=None: build_reg_mlp(input_dim=X_single.shape[1]),
    epochs=3, batch_size=64, verbose=0,
)

reg_results = runner_reg.run(model=reg_model, data=(X_single, y_future_idx), groups=patient_ids)

rows=[]
for i, (tr, te) in enumerate(reg_results.indices):
    y_true = y_future_idx[te]
    y_hat = np.asarray(reg_results.predictions[i]).reshape(-1)
    rows.append({
        "fold": i+1,
        "rmse": mean_squared_error(y_true, y_hat, squared=False),
        "r2": r2_score(y_true, y_hat),
    })
df_reg = pd.DataFrame(rows)
display(df_reg)
print(df_reg[["rmse","r2"]].agg(["mean","std"]))



## Step 7) Multi-input model (X is dict or list) with UniversalCVRunner + KerasSkWrap

We keep CV fully automatic by passing **indexable containers**:

- `X_dict_idx`: dict of arrays (clinical branch + imaging branch)
- `X_list_idx`: list of arrays (same info, different representation)

We evaluate MACE classification again, but with a true multi-input Keras model.


In [ ]:

def build_multiinput_model(input_shape=None, n_classes=None):
    # input_shape here can be a dict: {"clinical": (3,), "imaging": (2,)}
    inp_clin = layers.Input(shape=(3,), name="clinical")
    inp_img  = layers.Input(shape=(2,), name="imaging")

    x1 = layers.Dense(16, activation="relu")(inp_clin)
    x2 = layers.Dense(16, activation="relu")(inp_img)

    x = layers.Concatenate()([x1, x2])
    x = layers.Dense(16, activation="relu")(x)
    out = layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model({"clinical": inp_clin, "imaging": inp_img}, out)
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="binary_crossentropy",
                  metrics=["accuracy"])
    return model

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
runner_mi = UniversalCVRunner(cv_splitter=cv, framework="sklearn", verbose=1)

mi_clf = KerasSkWrap(
    build_fn=build_multiinput_model,
    epochs=3, batch_size=64, verbose=0,
    task="binary",
    threshold=0.5
)

mi_results = runner_mi.run(model=mi_clf, data=(X_dict_idx, y_main_idx), groups=patient_ids)

rows=[]
for i, (tr, te) in enumerate(mi_results.indices):
    y_true = y_main_idx[te]
    y_prob = mi_results.probabilities[i][:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    rows.append({
        "fold": i+1,
        "accuracy": accuracy_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_prob),
    })
df_mi = pd.DataFrame(rows)
display(df_mi)
print(df_mi[["accuracy","roc_auc"]].agg(["mean","std"]))



## Step 8) Multi-output model (classification + regression heads)

We train one Keras model with two heads:
- `mace` (sigmoid)  
- `ef_drop` (linear)

Then we run CV twice with the **same build function**:
1) evaluate the **classification head** using `KerasSkWrap(output_key="mace")`
2) evaluate the **regression head** using `KerasRegressorWrap(output_key="ef_drop")`

This demonstrates that TrustCV wrappers can **train multi-output models** while selecting the evaluated head via `output_key`.


In [ ]:

def build_multioutput_model():
    # single-array input for simplicity
    inp = layers.Input(shape=(X_single.shape[1],), name="features")
    x = layers.Dense(32, activation="relu")(inp)
    x = layers.Dropout(0.2)(x)

    mace = layers.Dense(1, activation="sigmoid", name="mace")(x)
    ef_drop = layers.Dense(1, activation="linear", name="ef_drop")(x)

    model = keras.Model(inp, {"mace": mace, "ef_drop": ef_drop})
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss={"mace": "binary_crossentropy", "ef_drop": "mse"},
        loss_weights={"mace": 1.0, "ef_drop": 0.3},
        metrics={"mace": ["accuracy"]},
    )
    return model

cv_g = GroupKFold(n_splits=3)
runner_mo = UniversalCVRunner(cv_splitter=cv_g, framework="sklearn", verbose=1)

# 8A) classification head
mo_clf = KerasSkWrap(
    build_fn=lambda input_shape=None, n_classes=None: build_multioutput_model(),
    epochs=3, batch_size=64, verbose=0,
    task="binary",
    threshold=0.5,
    output_key="mace",  # select classification head for predict/predict_proba
)

mo_clf_results = runner_mo.run(model=mo_clf, data=(X_single, y_multiout), groups=patient_ids)

rows=[]
for i, (tr, te) in enumerate(mo_clf_results.indices):
    y_true = y_main_idx[te]
    y_prob = mo_clf_results.probabilities[i][:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    rows.append({
        "fold": i+1,
        "accuracy": accuracy_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "head": "mace",
    })
df_mo_clf = pd.DataFrame(rows)
display(df_mo_clf)
print(df_mo_clf[["accuracy","roc_auc"]].agg(["mean","std"]))

# 8B) regression head
mo_reg = KerasRegressorWrap(
    build_fn=lambda input_shape=None, n_classes=None: build_multioutput_model(),
    epochs=3, batch_size=64, verbose=0,
    output_key="ef_drop",  # select regression head for predict
)

mo_reg_results = runner_mo.run(model=mo_reg, data=(X_single, y_multiout), groups=patient_ids)

rows=[]
for i, (tr, te) in enumerate(mo_reg_results.indices):
    y_true = y_future_idx[te]
    y_hat = np.asarray(mo_reg_results.predictions[i]).reshape(-1)
    rows.append({
        "fold": i+1,
        "rmse": mean_squared_error(y_true, y_hat, squared=False),
        "r2": r2_score(y_true, y_hat),
        "head": "ef_drop",
    })
df_mo_reg = pd.DataFrame(rows)
display(df_mo_reg)
print(df_mo_reg[["rmse","r2"]].agg(["mean","std"]))



## Step 9) Custom training loop (`train_step`) + `tf.data.Dataset` input

Some research code uses custom `train_step` (e.g., to log extra signals, handle special losses, or use sample weights).

Here we:
- build a small custom Model subclass
- feed data as a `tf.data.Dataset` (through an indexable dataset wrapper so CV splitting remains automatic)
- run group-safe CV with `UniversalCVRunner`


In [ ]:

class CustomTrainStepModel(keras.Model):
    def __init__(self, input_dim):
        super().__init__()
        self.d1 = layers.Dense(32, activation="relu")
        self.d2 = layers.Dense(16, activation="relu")
        self.out = layers.Dense(1, activation="sigmoid")

        self.loss_tracker = keras.metrics.Mean(name="loss")
        self.acc = keras.metrics.BinaryAccuracy(name="acc")

    def call(self, x, training=False):
        x = self.d1(x)
        x = self.d2(x)
        return self.out(x)

    @property
    def metrics(self):
        return [self.loss_tracker, self.acc]

    def train_step(self, data):
        # supports (x,y) or (x,y,sample_weight)
        if len(data) == 3:
            x, y, sw = data
        else:
            x, y = data
            sw = None

        with tf.GradientTape() as tape:
            y_pred = self(x, training=True)
            loss = self.compiled_loss(y, y_pred, sample_weight=sw, regularization_losses=self.losses)

        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))

        self.loss_tracker.update_state(loss)
        self.acc.update_state(y, y_pred, sample_weight=sw)
        return {"loss": self.loss_tracker.result(), "acc": self.acc.result()}

def build_custom_model(input_shape=None, n_classes=None):
    m = CustomTrainStepModel(input_dim=X_single.shape[1])
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="binary_crossentropy")
    return m

# sample weights example (e.g., upweight minority class)
sw = np.where(y_main_idx == 1, 2.0, 1.0).astype("float32")

# make indexable datasets per fold
X_ds_source = X_single.astype("float32")
y_ds_source = y_main_idx.astype("float32")

ds_indexable = IndexableTFDataset(
    X=X_ds_source,
    y=y_ds_source,
    sample_weight=sw,
    batch_size=64,
    shuffle=True
)

cv_g = GroupKFold(n_splits=3)
runner_ds = UniversalCVRunner(cv_splitter=cv_g, framework="sklearn", verbose=1)

ds_clf = KerasSkWrap(
    build_fn=lambda input_shape=None, n_classes=None: build_custom_model(),
    epochs=3,
    batch_size=64,
    verbose=0,
    task="binary",
    threshold=0.5,
    allow_dataset=True
)

ds_results = runner_ds.run(model=ds_clf, data=(ds_indexable, y_main_idx), groups=patient_ids)

rows=[]
for i, (tr, te) in enumerate(ds_results.indices):
    y_true = y_main_idx[te]
    y_prob = ds_results.probabilities[i][:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    rows.append({
        "fold": i+1,
        "accuracy": accuracy_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_prob),
    })
df_ds = pd.DataFrame(rows)
display(df_ds)
print(df_ds[["accuracy","roc_auc"]].agg(["mean","std"]))



# What this notebook demonstrates

1) **Leakage detection**: `TrustCVValidator` can flag group leakage if you pick a splitter that ignores groups.  
2) **Fold-level reporting**: `UniversalCVRunner` provides per-fold indices/predictions so you can compute any metrics (including multilabel/regression).  
3) **Deep learning interoperability**: `KerasSkWrap` / `KerasRegressorWrap` let Keras models behave like sklearn estimators for CV tooling.  
4) **Advanced DL patterns** work in CV:
   - multi-label (sigmoid C outputs)
   - regression
   - multi-input (dict)
   - multi-output (dict y, dict outputs, head selection via `output_key`)
   - custom `train_step` + dataset input (and sample weights)

If you want, I can also convert this notebook into a **minimal docs page** for your repo and add a small automated check (pytest) that runs a subset of these scenarios.
